# 第 2 周 · 第 1 天作业 —— 三方对话（GPT × Gemini × 本地 Ollama）

## 练习目标

让三个模型扮演不同角色，轮流发言，形成一场「多角色聊天」：

- **GPT-5-nano** → Alex（JS 开发者）
- **Gemini-3.1-flash-lite-preview** → Blake（Ruby 开发者）
- **本地 llama3.2（Ollama）** → Charlie（全栈，提供代码片段）

选用廉价 / 小模型，把 API 成本压到最低；Ollama 走本机，不产生云端费用。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI 兼容端点 | 同一套 `OpenAI` 客户端，改 `base_url` 打 Gemini / Ollama |
| System / User messages | 每人独立的 system 人设 + 拼接历史对话的 user prompt |
| 多轮对话 | 三个消息列表同步 append，循环 5 轮 |

## 怎么跑

1. `.env` 里准备好 `OPENAI_API_KEY`、`GOOGLE_API_KEY`
2. 本机 `ollama serve`，并已 `ollama pull llama3.2`
3. 自上而下运行各单元格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 标准库 requests：本练习导入后未直接使用，保留原样以免改依赖结构
import requests
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：Chat Completions 的统一入口（也可指向兼容端点）
from openai import OpenAI
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮地展示对话
from IPython.display import Markdown, display


In [ ]:
# ========== 环境：加载密钥并做存在性检查 ==========

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量取出 OpenAI 密钥（字符串本身不改）
openai_api_key = os.getenv('OPENAI_API_KEY')
# 从环境变量取出 Google（Gemini）密钥
google_api_key = os.getenv('GOOGLE_API_KEY')

# 若密钥存在：打印前缀做「脱敏确认」，方便排查配置问题
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# 同样检查 Google API Key（只显示前 2 个字符）
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")


In [ ]:
# ========== 客户端 + 模型名 + 人设 + 开场白 ==========

# 创建默认 OpenAI 客户端（读环境变量里的 OPENAI_API_KEY）
# 围绕 HTTP 端点的薄包装：chat.completions.create(...)
openai = OpenAI()

# Gemini 与 Ollama 都提供 OpenAI 兼容端点，因此可复用同一客户端类
# 更换 base_url 即可把请求打到不同后端
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

# Gemini 客户端：api_key 用 Google 密钥，base_url 指向 Google 兼容网关
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
# Ollama 客户端：本地服务通常用占位 api_key="ollama"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# 三个角色各自使用的模型 id（字符串必须与后端已注册名一致，勿擅自改）
gpt_model = "gpt-5-nano"
gemini_model = "gemini-3.1-flash-lite-preview"
ollama_model = "llama3.2"

# System prompt：定义 GPT 侧角色 Alex（JS 开发者）—— 影响行为的英文原文保留
gpt_system = """
    You are Alex, a JS Developer.
    You are in a conversation with Blake and Charlie.
    You always ask what the Ruby code does to Blake.
    """

# System prompt：Gemini 侧角色 Blake（Ruby 专家）
gemini_system = """
    You are Blake, a Ruby Developer and you are happy to explain Ruby concepts to beginners.
    You are in a conversation with Alex and Charlie.
"""

# System prompt：Ollama 侧角色 Charlie（全栈，提供代码示例）
ollama_system = """
    You are Charlie, a Fullstack Developer and provide code snippets examples
    when anyone talks about the Ruby programming language.
    You are in a conversation with Alex and Blake.
"""

# 各角色的初始发言列表（后续每轮 append 新回复）
gpt_messages = ["""
    Hello, nice to meet you! I'm Alex and I want to learn more about Ruby.
"""]
gemini_messages = ["""
    Hello, nice to meet you too! I'm Blake and I'm a Ruby expert.
    Could you provide a code snippet so that I can explain it to Alex?
"""]
ollama_messages = ["""
    Hello, nice to meet you too! I'm Charlie and I can provide a new code snippet everytime you ask.
    How about we start a simple 'Hello World' example:
    ```ruby
    puts "Hello, World!"
    ```
"""]


In [ ]:
# ========== 三个调用函数：拼历史 → 带人设提问 → 返回下一句 ==========

# 让 GPT（Alex）根据目前三人对话历史生成下一句
def call_gpt():
    # messages 以 system 人设开头
    messages = [{"role": "system", "content": gpt_system}]
    # conversation：把三方历史拼成一段可读摘要，塞进 user prompt
    conversation = ""
    # zip 同步遍历三个消息列表（按轮次对齐）
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        # 原逻辑条件保留（变量名 gpt/gemini/ollama 勿改）
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    # user prompt：告诉模型「你是 Alex，请说下一句」（英文原文保留）
    user_prompt = f"""
        You are Alex, in a conversation with Blake and Charlie.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Alex.
    """
    # 把 user 消息追加进 messages
    messages.append({"role": "user", "content": user_prompt})
    # 调用云端 GPT 模型
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    # 取出助手回复文本
    return response.choices[0].message.content

# 让 Gemini（Blake）生成下一句；user 里额外带上 Alex 最新一句
def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    conversation = ""
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    user_prompt = f"""
        You are Blake, in a conversation with Alex and Charlie.
        The conversation so far is as follows:
        {conversation}
        Alex: {gpt_messages[-1]}
        Now with this, respond with what you would like to say next, as Blake.
    """
    messages.append({"role": "user", "content": user_prompt})
    # 走 gemini 客户端 + gemini_model
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

# 让 Charlie 生成下一句；user 里带上 Alex、Blake 最新一句
def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]
    conversation = ""
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    user_prompt = f"""
        You are Charlie, in a conversation with Alex and Blake.
        The conversation so far is as follows:
        {conversation}
        Alex: {gpt_messages[-1]}
        Blake: {gemini_messages[-1]}
        Now with this, respond with what you would like to say next, as Charlie.
    """
    messages.append({"role": "user", "content": user_prompt})
    # 原代码此处调用 gemini 客户端与 gemini_model（保持原样，不「修正」）
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# ========== 执行：先展示开场白，再循环 5 轮三方发言 ==========

# 先把三个角色的初始消息用 Markdown 显示出来
display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Ollama:\n{ollama_messages[0]}\n"))

# 再进行 5 轮：GPT → Gemini → Ollama，每轮展示并 append 到各自历史
for i in range(5):
    # GPT（Alex）发言
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    # Gemini（Blake）发言
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)

    # Charlie 一侧（函数名 call_ollama，实现按原代码）
    ollama_next = call_ollama()
    display(Markdown(f"### Ollama:\n{ollama_next}\n"))
    ollama_messages.append(ollama_next)
